In [2]:
import Pkg
Pkg.activate(dirname(@__DIR__)) #need design-specific dependencies

using FlightCore, FlightPhysics, FlightApps
using FlightPhysics.Linearization: delete_vars
using FlightPhysics.Control: PIDDataPoint, LQRDataPoint
using ControlSystems, RobustAndOptimalControl, Plots, ComponentArrays, LinearAlgebra

  Activating project at `~/.julia/dev/Flight.jl/lib/FlightApps/design`


ArgumentError: ArgumentError: Package FlightCore [e3cc1ee6-15ac-4fef-96b3-1947582bf537] is required but does not seem to be installed:
 - Run `Pkg.instantiate()` to install all recorded dependencies.


## 1. Open Loop Dynamics

In [3]:
mdl = Robot2D.Vehicle() |> Model
lss = linearize(mdl)
P = named_ss(lss)

UndefVarError: UndefVarError: `Robot2D` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [4]:
dampreport(P)

UndefVarError: UndefVarError: `dampreport` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## 2. Velocity Controller

In [5]:
#reduced system with η removed
lss_red = delete_vars(lss, :η)
P_red = named_ss(lss_red)

UndefVarError: UndefVarError: `delete_vars` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [6]:
P_v, params_v2m = let lss = lss_red

    x_trim = lss.x0
    n_x = length(x_trim)
    x_labels = collect(keys(x_trim))
    @assert tuple(x_labels...) === propertynames(Robot2D.XController())

    u_labels = [:m, ]
    u_trim = lss.u0[u_labels]

    z_labels = [:v, ]
    z_trim = lss.y0[z_labels]

    A = lss.A
    B = lss.B
    C = lss.C[z_labels, :]
    D = lss.D[z_labels, :]

    C_int = C[z_labels, :]
    D_int = D[z_labels, :]
    n_int = size(C_int, 1)

    A_aug = [A zeros(size(A, 1), n_int); C_int zeros(n_int, n_int)]
    B_aug = [B; D_int]
    C_aug = [C zeros(size(C, 1), n_int)]
    D_aug = D

    P_aug = ss(A_aug, B_aug, C_aug, D_aug)

    x_aug_labels = push!(copy(x_labels), :ξ_v)
    Q_diag = ComponentVector(zeros(length(x_aug_labels)), Axis(x_aug_labels))
    R_diag = ComponentVector(zeros(length(u_labels)), Axis(u_labels))

    Q_diag.ω = 1e-3
    Q_diag.v = 1e-2
    Q_diag.θ = 0
    Q_diag.ξ_v = 5e-2

    R_diag.m = 1e-1

    Q = diagm(Q_diag)
    R = diagm(R_diag)

    #compute augmented feedback matrix
    K_aug = lqr(P_aug, Q, R)

    L = [A B; C D]
    M = inv(L)
    M_12 = M[1:n_x, n_x+1:end]
    M_22 = M[n_x+1:end, n_x+1:end]

    #extract system state and integrator blocks from the feedback matrix
    K_x = K_aug[:, 1:n_x]
    K_ξ = K_aug[:, n_x+1:end]

    K_fbk = K_x
    K_fwd = M_22 + K_x * M_12
    K_int = K_ξ

    K_fbk_ss = named_ss(ss(K_fbk), u = x_labels, y = :m_fbk)
    K_fwd_ss = named_ss(ss(K_fwd), u = :v_ref, y = :m_fwd)
    K_int_ss = named_ss(ss(K_int), u = :v_err, y = :m_int_in)

    int_ss = named_ss(ss(tf(1, [1,0])) .* I(1),
                        x = :m_ξ,
                        u = :m_int_in,
                        y = :m_int_out);

    v_sum = sumblock("v_err = v - v_ref")
    m_sum = sumblock("m = m_fwd - m_fbk - m_int_out")

    connections = vcat(
        Pair.(x_labels, x_labels),
        :v_err => :v_err,
        :m_int_in=> :m_int_in,
        :m_int_out => :m_int_out,
        :m_fwd => :m_fwd,
        :m_fbk => :m_fbk,
        :m => :m,
        )

        #connect back to full plant
    P_v = connect([P, int_ss, K_fwd_ss, K_fbk_ss, K_int_ss,
                    v_sum, m_sum], connections;
                    w1 = :v_ref, z1 = P.y, unique = false)

    params = LQRDataPoint(;
        K_fbk = Matrix(K_fbk), K_fwd = Matrix(K_fwd), K_int = Matrix(K_int),
        x_trim = Vector(x_trim), u_trim = Vector(u_trim), z_trim = Vector(z_trim))

    (P_v, params)

end

nothing

UndefVarError: UndefVarError: `lss_red` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [7]:
dampreport(P_v)

UndefVarError: UndefVarError: `dampreport` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [8]:
t_sim = 5
step(P_v[:v, :v_ref], t_sim) |> stepinfo |> display
step(P_v[:v, :v_ref], t_sim) |> plot
step(P_v[:u_m, :v_ref], t_sim) |> plot!
step(P_v[:ω, :v_ref], t_sim) |> plot!
step(P_v[:θ, :v_ref], t_sim) |> plot!
# step(P_v[:η, :v_ref], t_sim) |> plot!

UndefVarError: UndefVarError: `P_v` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [9]:
#simulate a unit velocity command input applied from t=1 to t = 5
y, t, _, _ = lsim(P_v[:u_m, :v_ref], (x, t)->[1.0]*(1<=t<10), 0:0.01:20)
plot(t, vec(y); label = "Linear")

UndefVarError: UndefVarError: `lsim` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

This shows that stopping the vehicle requires a transient increase in motor input.
Therefore, to ensure we always leave sufficient control authority to stop the vehicle, we should
limit the velocity reference input to a fraction of the maximum steady state
velocity, that is, the steady state velocity corresponding to the maximum motor input.

## 3. Position Controller

In [10]:
P_v2η = P_v[:η, :v_ref]
step(P_v2η, t_sim) |> plot

UndefVarError: UndefVarError: `P_v` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [11]:
marginplot(P_v2η)

UndefVarError: UndefVarError: `marginplot` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
nyquistplot(P_v2η; unit_circle = true, Ms_circles = [1.0])
plot!(ylims = (-2,2), xlims = (-2,2))

UndefVarError: UndefVarError: `nyquistplot` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [13]:
k_p_η2v = 0.6
C_η2v = named_ss(ss(k_p_η2v), :C_η2v; u = :η_err, y = :v_ref);
L_η2v = series(C_η2v, P_v2η);

UndefVarError: UndefVarError: `ss` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [14]:
marginplot(L_η2v)

UndefVarError: UndefVarError: `marginplot` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [15]:
#closed loop transfer function
T_η2v = output_comp_sensitivity(P_v2η, C_η2v)
T_η2v_step_SISO = step(T_η2v, 2t_sim)
stepinfo(T_η2v_step_SISO) |> display
T_η2v_step_SISO |> plot

UndefVarError: UndefVarError: `output_comp_sensitivity` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [16]:
#connect the position loop
η2v_sum = sumblock("η_err = η_ref - η")
P_η = connect([P_v, η2v_sum, C_η2v], [:η_err=>:η_err, :η=>:η, :v_ref=>:v_ref], w1 = [:η_ref, ], z1 = P_v.y)

#verify the response matches that of the SISO closed loop
step(P_η[:η, :η_ref], 2t_sim) |> plot
T_η2v_step_SISO |> plot!

UndefVarError: UndefVarError: `sumblock` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [17]:
step(P_η[:η, :η_ref], 2t_sim) |> plot
step(P_η[:v, :η_ref], 2t_sim) |> plot!
step(P_η[:u_m, :η_ref], t_sim) |> plot!
# step(P_η[:ω, :η_ref], t_sim) |> plot!
# step(P_η[:θ, :η_ref], t_sim) |> plot!

UndefVarError: UndefVarError: `P_η` not defined in `Main`
Suggestion: check for spelling errors or missing imports.